# 02 - Data Cleaning

## Goal

Clean and validate the raw Ticketmaster event data collected during API ingestion.

## Tasks

- Load the raw JSON data
- Flatten nested event data
- Investigate duplicate events
- Handle missing values
- Validate dates and times
- Convert geographic fields to correct types
- Standardize text fields
- Save the cleaned dataset

In [3]:
import pandas as pd
import json
from pathlib import Path

In [6]:
project_path = Path("..")
raw_path = (
    project_path / "data" / "raw" / "ticketmaster" / "germany_music_events.json"
    )

with open(raw_path, "r", encoding= "utf-8") as file:
    raw_events = json.load(file)

In [ ]:
type(raw_events)
len(raw_events)

list

## Flatten Raw Events

The Ticketmaster API returns nested JSON data.  
The raw events are flattened into a tabular structure for cleaning and validation.

In [ ]:
def flatten_event(event):
    embedded = event.get("_embedded", {})

    venues = embedded.get("venues", [])
    venue = venues[0] if venues else {}

    attractions = embedded.get("attractions", [])
    artist = attractions[0] if attractions else {}

    location = venue.get("location", {})

    return {
        "event_id": event.get("id"),
        "event_name": event.get("name"),
        "artist_name": artist.get("name"),
        "event_date": event.get("dates", {}).get("start", {}).get("localDate"),
        "event_time": event.get("dates", {}).get("start", {}).get("localTime"),
        "venue_name": venue.get("name"),
        "city": venue.get("city", {}).get("name"),
        "country": venue.get("country", {}).get("name"),
        "latitude": location.get("latitude"),
        "longitude": location.get("longitude"),
        "event_url": event.get("url")
    }
    

In [ ]:
flatten_event(raw_events[0
])

{'event_id': 'LvZ18QLUFcKuwNYZ0XXWn',
 'event_name': 'Heavysaurus - METAL Tour 2026',
 'artist_name': 'Heavysaurus',
 'event_date': '2026-08-30',
 'event_time': '13:00:00',
 'venue_name': 'Schön & Frölich',
 'city': 'Braunschweig',
 'country': 'Germany',
 'latitude': '52.25654',
 'longitude': '10.49957',
 'event_url': 'https://www.universe.com/events/heavysaurus-metal-tour-2026-tickets-X4HCW6?ref=ticketmaster'}

In [13]:
rows = [
    flatten_event(event)
    for event in raw_events
]

events_df = pd.DataFrame(rows)

In [15]:
events_df.head()

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
0,LvZ18QLUFcKuwNYZ0XXWn,Heavysaurus - METAL Tour 2026,Heavysaurus,2026-08-30,13:00:00,Schön & Frölich,Braunschweig,Germany,52.25654,10.49957,https://www.universe.com/events/heavysaurus-me...
1,Z698xZC2Z16v8KeAfJ,Melanie Martinez – HADES: THE SACRIFICE | VIP,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
2,Z698xZC2Z1kAIGIZg,Melanie Martinez – HADES: THE SACRIFICE,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
3,LvZ18Qpz8oKu0POZyE61A,SUPERBLOOM 2026 Experience Day - Sonntag,SUPERBLOOM Festival,2026-08-30,10:00:00,NaN,Munich,Germany,48.17429,11.55524,https://www.universe.com/events/superbloom-202...
4,Z698xZC2Z1kCpjv8P,ITZY 3RD WORLD TOUR <TUNNEL VISION> in FRANKFURT,ITZY,2026-09-17,19:30:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/itzy-3rd-wor...


## Duplicate Events

Duplicate event IDs are investigated before removal to ensure that repeated records do not contain conflicting information.

In [20]:
duplicate_events = events_df[
    events_df["event_id"].duplicated(keep= False)
].sort_values("event_id")

duplicate_events

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
943,LvZ18QNf90GZCmOvG95vu,FRITZ KALKBRENNER,Fritz Kalkbrenner,2026-12-19,22:00:00,KARREE,Freiburg,Germany,47.99808,7.84851,https://www.universe.com/events/fritz-kalkbren...
852,LvZ18QNf90GZCmOvG95vu,FRITZ KALKBRENNER,Fritz Kalkbrenner,2026-12-19,22:00:00,KARREE,Freiburg,Germany,47.99808,7.84851,https://www.universe.com/events/fritz-kalkbren...
1059,LvZ18QQLjjSI3-YZ0gwJ5,BOND NIGHT mit Dennis Durant & Band im Hotel A...,DWEDA Records,2027-02-19,19:00:00,NaN,Hamburg,Germany,53.55751,10.00524,https://www.universe.com/events/bond-night-mit...
1013,LvZ18QQLjjSI3-YZ0gwJ5,BOND NIGHT mit Dennis Durant & Band im Hotel A...,DWEDA Records,2027-02-19,19:00:00,NaN,Hamburg,Germany,53.55751,10.00524,https://www.universe.com/events/bond-night-mit...
491,Z698xZC2Z16v4fG4uE,Hatsune Miku - MIKU EXPO 2026 EUROPE,Hatsune Miku,2026-11-17,20:00:00,NaN,Berlin,Germany,52.53117,13.45067,https://www.ticketmaster.de/event/hatsune-miku...
508,Z698xZC2Z16v4fG4uE,Hatsune Miku - MIKU EXPO 2026 EUROPE,Hatsune Miku,2026-11-17,20:00:00,NaN,Berlin,Germany,52.53117,13.45067,https://www.ticketmaster.de/event/hatsune-miku...
981,Z698xZC2Z16v4s8xP4,Unheilig | Box seat in the Ticketmaster Suite,Unheilig,2027-01-29,19:45:00,Barclays Arena,Hamburg,Germany,53.58894,9.899,https://www.ticketmaster.de/event/unheilig-%7C...
983,Z698xZC2Z16v4s8xP4,Unheilig | Box seat in the Ticketmaster Suite,Unheilig,2027-01-29,19:45:00,Barclays Arena,Hamburg,Germany,53.58894,9.899,https://www.ticketmaster.de/event/unheilig-%7C...
222,Z698xZC2Z16vOyPJvp,Bryan Adams - Roll With The Punches Tour,Bryan Adams,2026-10-12,20:00:00,SAP Arena,Mannheim,Germany,49.46424,8.51797,https://www.ticketmaster.de/event/bryan-adams-...
217,Z698xZC2Z16vOyPJvp,Bryan Adams - Roll With The Punches Tour,Bryan Adams,2026-10-12,20:00:00,SAP Arena,Mannheim,Germany,49.46424,8.51797,https://www.ticketmaster.de/event/bryan-adams-...


In [23]:
duplicate_events["event_id"].nunique()

9

In [ ]:
duplicate_comparison = (
    duplicate_events.groupby("event_id").nunique(dropna= False)
)
duplicate_comparison

In [33]:
events_df = (
    events_df.drop_duplicates(subset= "event_id", keep= "first").reset_index(drop= True)
)

events_df.head()

,event_id,event_name,artist_name,event_date,event_time,venue_name,city,country,latitude,longitude,event_url
0,LvZ18QLUFcKuwNYZ0XXWn,Heavysaurus - METAL Tour 2026,Heavysaurus,2026-08-30,13:00:00,Schön & Frölich,Braunschweig,Germany,52.25654,10.49957,https://www.universe.com/events/heavysaurus-me...
1,Z698xZC2Z16v8KeAfJ,Melanie Martinez – HADES: THE SACRIFICE | VIP,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
2,Z698xZC2Z1kAIGIZg,Melanie Martinez – HADES: THE SACRIFICE,Melanie Martinez,2026-09-18,20:00:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/melanie-mart...
3,LvZ18Qpz8oKu0POZyE61A,SUPERBLOOM 2026 Experience Day - Sonntag,SUPERBLOOM Festival,2026-08-30,10:00:00,NaN,Munich,Germany,48.17429,11.55524,https://www.universe.com/events/superbloom-202...
4,Z698xZC2Z1kCpjv8P,ITZY 3RD WORLD TOUR <TUNNEL VISION> in FRANKFURT,ITZY,2026-09-17,19:30:00,NaN,Frankfurt am Main,Germany,50.11233,8.65073,https://www.ticketmaster.de/event/itzy-3rd-wor...


In [34]:
print("Rows after duplicate removal:", len(events_df))
print("Duplicate event IDs:", events_df["event_id"].duplicated().sum())
print("Event IDs are unique:", events_df["event_id"].is_unique)

Rows after duplicate removal: 1172
Duplicate event IDs: 0
Event IDs are unique: True
